In [1]:
!pip install catboost -q
from google.colab import drive; drive.mount('/content/drive')

import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings("ignore")

# Folder with train.csv / test.csv / submission.csv (auto-detects Colab Drive path)
DATA_DIR = "/content/drive/MyDrive/ML_project_dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/mnt/user-data/uploads"
OUT_DIR = DATA_DIR if DATA_DIR.startswith("/content") else "/mnt/user-data/outputs"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/submission.csv")
print(train.shape, test.shape, sample_sub.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.9 MB/s eta 0:00:00
Mounted at /content/drive
(18076, 12) (4520, 11) (4520, 2)


In [2]:
def add_route(d):
    d = d.copy()
    d["route"] = d["source_city"].astype(str) + "_to_" + d["destination_city"].astype(str)
    return d

FEATS = ["airline","source_city","departure_time","stops","arrival_time","destination_city",
         "flight_class","duration","days_left","flight","route"]

X  = add_route(train)[FEATS].reset_index(drop=True)
y  = train["price"].reset_index(drop=True)
Xt = add_route(test)[FEATS].reset_index(drop=True)
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(X[c])]

# category levels = union of train+test values (only the *list of names*, no price information)
LEVELS = {c: sorted(set(X[c]) | set(Xt[c])) for c in CATS}

print("categorical:", CATS)
print("train features:", X.shape, "| test features:", Xt.shape)

categorical: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'flight', 'route']
train features: (18076, 11) | test features: (4520, 11)


In [3]:
def as_cat(df):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=LEVELS[c])
    return df

def make_model(name, seed=42):
    if name == "CatBoost d8":
        return CatBoostRegressor(iterations=800, depth=8, learning_rate=0.06, random_seed=seed,
                                 verbose=0, cat_features=CATS, thread_count=-1)
    if name == "CatBoost d6":
        return CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.06, l2_leaf_reg=3,
                                 random_seed=seed, verbose=0, cat_features=CATS, thread_count=-1)
    if name == "XGBoost":
        return xgb.XGBRegressor(n_estimators=2000, learning_rate=0.015, max_depth=8, min_child_weight=3,
                                reg_lambda=40, subsample=0.7, colsample_bytree=0.5, colsample_bynode=0.6,
                                tree_method="hist", enable_categorical=True, max_cat_to_onehot=4,
                                max_cat_threshold=64, random_state=seed, n_jobs=-1)
    if name == "LightGBM":
        return lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, min_child_samples=10,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, cat_smooth=20,
                                 verbose=-1, random_state=seed)

def fit_predict(name, X_tr, y_tr, X_te):
    m = make_model(name)
    if name.startswith("CatBoost"):
        m.fit(X_tr, y_tr); return m.predict(X_te)
    m.fit(as_cat(X_tr), y_tr); return m.predict(as_cat(X_te))

def metrics(y_true, p):
    return dict(RMSE=mean_squared_error(y_true, p)**.5, MAE=mean_absolute_error(y_true, p), R2=r2_score(y_true, p))

MODELS = ["CatBoost d8", "CatBoost d6", "XGBoost", "LightGBM"]

In [4]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

rows, val_pred = [], {}
for name in MODELS:
    t = time.time()
    val_pred[name] = fit_predict(name, X_train, y_train, X_val)
    rows.append({"Model": name, **metrics(y_val, val_pred[name]), "sec": round(time.time()-t)})

val_table = pd.DataFrame(rows).set_index("Model").round(4)
val_table.loc["Random Forest (earlier notebook)"] = [144.739, 47.017, 0.6276, np.nan]   # recorded result
val_table.loc["Poly degree 8 (original baseline)"] = [176.260, 86.435, 0.4477, np.nan]   # recorded result
val_table

,RMSE,MAE,R2,sec
Model,,,,
CatBoost d8,140.7829,49.3047,0.6477,40.0
CatBoost d6,141.5311,50.7065,0.6439,25.0
XGBoost,141.2646,47.0346,0.6453,17.0
LightGBM,144.1350,49.7454,0.6307,3.0
Random Forest (earlier notebook),144.7390,47.0170,0.6276,NaN
Poly degree 8 (original baseline),176.2600,86.4350,0.4477,NaN


In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(X))
oof = {}
for name in MODELS:
    t = time.time(); o = np.zeros(len(X))
    for tr_i, va_i in folds:
        o[va_i] = fit_predict(name, X.iloc[tr_i], y.iloc[tr_i], X.iloc[va_i])
    oof[name] = o
    print(f"{name:12s} done in {time.time()-t:.0f}s")

cv_rows = []
for name, o in oof.items():
    fold_r2 = [r2_score(y.iloc[v], o[v]) for _, v in folds]
    cv_rows.append({"Model": name, **metrics(y, o), **{f"fold{i+1}": r for i, r in enumerate(fold_r2)}})
cv_table = pd.DataFrame(cv_rows).set_index("Model").round(4)
cv_table

CatBoost d8  done in 174s
CatBoost d6  done in 125s
XGBoost      done in 90s
LightGBM     done in 14s


,RMSE,MAE,R2,fold1,fold2,fold3,fold4,fold5
Model,,,,,,,,
CatBoost d8,122.8713,47.5475,0.6936,0.6452,0.7795,0.7782,0.7465,0.5897
CatBoost d6,123.3718,48.8076,0.6911,0.6457,0.7716,0.7781,0.7500,0.5828
XGBoost,123.5410,45.3110,0.6902,0.6465,0.7884,0.7678,0.7772,0.5578
LightGBM,125.8401,47.4719,0.6786,0.6333,0.7767,0.7504,0.7458,0.5637


In [6]:
WEIGHTS = {"CatBoost d8": 0.35, "CatBoost d6": 0.20, "XGBoost": 0.30, "LightGBM": 0.15}

def blend(preds, w=WEIGHTS):
    return sum(w[k] * preds[k] for k in w)

blend_oof = blend(oof)
rows = [{"Model": "Blend (4 models)", **metrics(y, blend_oof)}]
rows.append({"Model": "CatBoost d8 + XGBoost 50/50", **metrics(y, .5*oof["CatBoost d8"] + .5*oof["XGBoost"])})
rows.append({"Model": "Old blend (2 CatBoost + LightGBM)",
             **metrics(y, .4*oof["CatBoost d8"] + .3*oof["CatBoost d6"] + .3*oof["LightGBM"])})
rows.append({"Model": "CatBoost d8 alone", **metrics(y, oof["CatBoost d8"])})
rows.append({"Model": "XGBoost alone", **metrics(y, oof["XGBoost"])})
blend_table = pd.DataFrame(rows).set_index("Model").round(4)
print(blend_table)

print("\nPer-fold R2, blend:", [round(r2_score(y.iloc[v], blend_oof[v]), 3) for _, v in folds])
print("Residual correlation between models (lower = more useful to blend):")
print(pd.DataFrame({k: y - v for k, v in oof.items()}).corr().round(3))

                                       RMSE      MAE      R2
Model                                                       
Blend (4 models)                   120.9696  45.0312  0.7030
CatBoost d8 + XGBoost 50/50        121.1308  44.6756  0.7022
Old blend (2 CatBoost + LightGBM)  121.7812  46.4170  0.6990
CatBoost d8 alone                  122.8713  47.5475  0.6936
XGBoost alone                      123.5410  45.3110  0.6902

Per-fold R2, blend: [0.651, 0.798, 0.785, 0.78, 0.583]
Residual correlation between models (lower = more useful to blend):
             CatBoost d8  CatBoost d6  XGBoost  LightGBM
CatBoost d8        1.000        0.989    0.933     0.928
CatBoost d6        0.989        1.000    0.923     0.922
XGBoost            0.933        0.923    1.000     0.941
LightGBM           0.928        0.922    0.941     1.000


In [7]:
test_pred = {}
for name in MODELS:
    t = time.time()
    test_pred[name] = fit_predict(name, X, y, Xt)
    print(f"{name:12s} fitted on full data ({time.time()-t:.0f}s)")

final = np.clip(blend(test_pred), 0, None)

submission = sample_sub.copy()
submission["price"] = final
assert submission.shape == (len(test), 2) and list(submission.columns) == ["id", "price"]
assert submission["id"].tolist() == test["id"].tolist(), "ID order changed!"
assert submission["price"].notna().all()

SAVE_PATH = f"{OUT_DIR}/submission_xgb_catboost_blend.csv"
submission.to_csv(SAVE_PATH, index=False)
print("saved:", SAVE_PATH)
print(submission["price"].describe().round(2))

CatBoost d8  fitted on full data (39s)
CatBoost d6  fitted on full data (29s)
XGBoost      fitted on full data (21s)
LightGBM     fitted on full data (3s)
saved: /content/drive/MyDrive/ML_project_dataset/submission_xgb_catboost_blend.csv
count    4520.00
mean      280.37
std       188.25
min        13.71
25%       158.41
50%       246.90
75%       347.44
max      2235.04
Name: price, dtype: float64


**EXPERIMENTING ON 5 FOLD **

In [8]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings("ignore")

# Folder with train.csv / test.csv / submission.csv (auto-detects Colab Drive path)
DATA_DIR = "/content/drive/MyDrive/ML_project_dataset"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "/mnt/user-data/uploads"
OUT_DIR = DATA_DIR if DATA_DIR.startswith("/content") else "/mnt/user-data/outputs"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/submission.csv")
print(train.shape, test.shape, sample_sub.shape)

(18076, 12) (4520, 11) (4520, 2)


In [9]:
def add_route(d):
    d = d.copy()
    d["route"] = d["source_city"].astype(str) + "_to_" + d["destination_city"].astype(str)
    return d

FEATS = ["airline","source_city","departure_time","stops","arrival_time","destination_city",
         "flight_class","duration","days_left","flight","route"]

X  = add_route(train)[FEATS].reset_index(drop=True)
y  = train["price"].reset_index(drop=True)
Xt = add_route(test)[FEATS].reset_index(drop=True)
CATS = [c for c in FEATS if not pd.api.types.is_numeric_dtype(X[c])]

# category levels = union of train+test values (only the *list of names*, no price information)
LEVELS = {c: sorted(set(X[c]) | set(Xt[c])) for c in CATS}
print("categorical:", CATS)
print("train features:", X.shape, "| test features:", Xt.shape)

categorical: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'flight_class', 'flight', 'route']
train features: (18076, 11) | test features: (4520, 11)


In [10]:
def as_cat(df):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=LEVELS[c])
    return df

def make_model(name, seed=42):
    if name == "CatBoost d8":
        return CatBoostRegressor(iterations=800, depth=8, learning_rate=0.06, random_seed=seed,
                                 verbose=0, cat_features=CATS, thread_count=-1)
    if name == "CatBoost d6":
        return CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.06, l2_leaf_reg=3,
                                 random_seed=seed, verbose=0, cat_features=CATS, thread_count=-1)
    if name == "XGBoost":
        return xgb.XGBRegressor(n_estimators=2000, learning_rate=0.015, max_depth=8, min_child_weight=3,
                                reg_lambda=40, subsample=0.7, colsample_bytree=0.5, colsample_bynode=0.6,
                                tree_method="hist", enable_categorical=True, max_cat_to_onehot=4,
                                max_cat_threshold=64, random_state=seed, n_jobs=-1)
    if name == "LightGBM":
        return lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=31, min_child_samples=10,
                                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8, cat_smooth=20,
                                 verbose=-1, random_state=seed)

def fit_predict(name, X_tr, y_tr, X_te):
    m = make_model(name)
    if name.startswith("CatBoost"):
        m.fit(X_tr, y_tr); return m.predict(X_te)
    m.fit(as_cat(X_tr), y_tr); return m.predict(as_cat(X_te))

def metrics(y_true, p):
    return dict(RMSE=mean_squared_error(y_true, p)**.5, MAE=mean_absolute_error(y_true, p), R2=r2_score(y_true, p))

MODELS = ["CatBoost d8", "CatBoost d6", "XGBoost", "LightGBM"]

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42)

rows, val_pred = [], {}
for name in MODELS:
    t = time.time()
    val_pred[name] = fit_predict(name, X_train, y_train, X_val)
    rows.append({"Model": name, **metrics(y_val, val_pred[name]), "sec": round(time.time()-t)})

val_table = pd.DataFrame(rows).set_index("Model").round(4)
val_table.loc["Random Forest (earlier notebook)"] = [144.739, 47.017, 0.6276, np.nan]   # recorded result
val_table.loc["Poly degree 8 (original baseline)"] = [176.260, 86.435, 0.4477, np.nan]   # recorded result
val_table

,RMSE,MAE,R2,sec
Model,,,,
CatBoost d8,140.7829,49.3047,0.6477,39.0
CatBoost d6,141.5311,50.7065,0.6439,26.0
XGBoost,141.2646,47.0346,0.6453,17.0
LightGBM,144.1350,49.7454,0.6307,2.0
Random Forest (earlier notebook),144.7390,47.0170,0.6276,NaN
Poly degree 8 (original baseline),176.2600,86.4350,0.4477,NaN


In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(X))
oof = {}
for name in MODELS:
    t = time.time(); o = np.zeros(len(X))
    for tr_i, va_i in folds:
        o[va_i] = fit_predict(name, X.iloc[tr_i], y.iloc[tr_i], X.iloc[va_i])
    oof[name] = o
    print(f"{name:12s} done in {time.time()-t:.0f}s")

cv_rows = []
for name, o in oof.items():
    fold_r2 = [r2_score(y.iloc[v], o[v]) for _, v in folds]
    cv_rows.append({"Model": name, **metrics(y, o), **{f"fold{i+1}": r for i, r in enumerate(fold_r2)}})
cv_table = pd.DataFrame(cv_rows).set_index("Model").round(4)
cv_table

CatBoost d8  done in 167s
CatBoost d6  done in 130s
XGBoost      done in 89s
LightGBM     done in 14s


,RMSE,MAE,R2,fold1,fold2,fold3,fold4,fold5
Model,,,,,,,,
CatBoost d8,122.8713,47.5475,0.6936,0.6452,0.7795,0.7782,0.7465,0.5897
CatBoost d6,123.3718,48.8076,0.6911,0.6457,0.7716,0.7781,0.7500,0.5828
XGBoost,123.5410,45.3110,0.6902,0.6465,0.7884,0.7678,0.7772,0.5578
LightGBM,125.8401,47.4719,0.6786,0.6333,0.7767,0.7504,0.7458,0.5637


In [13]:
WEIGHTS = {"CatBoost d8": 0.35, "CatBoost d6": 0.20, "XGBoost": 0.30, "LightGBM": 0.15}

def blend(preds, w=WEIGHTS):
    return sum(w[k] * preds[k] for k in w)

blend_oof = blend(oof)
rows = [{"Model": "Blend (4 models)", **metrics(y, blend_oof)}]
rows.append({"Model": "CatBoost d8 + XGBoost 50/50", **metrics(y, .5*oof["CatBoost d8"] + .5*oof["XGBoost"])})
rows.append({"Model": "Old blend (2 CatBoost + LightGBM)",
             **metrics(y, .4*oof["CatBoost d8"] + .3*oof["CatBoost d6"] + .3*oof["LightGBM"])})
rows.append({"Model": "CatBoost d8 alone", **metrics(y, oof["CatBoost d8"])})
rows.append({"Model": "XGBoost alone", **metrics(y, oof["XGBoost"])})
blend_table = pd.DataFrame(rows).set_index("Model").round(4)
print(blend_table)

print("\nPer-fold R2, blend:", [round(r2_score(y.iloc[v], blend_oof[v]), 3) for _, v in folds])
print("Residual correlation between models (lower = more useful to blend):")
print(pd.DataFrame({k: y - v for k, v in oof.items()}).corr().round(3))

                                       RMSE      MAE      R2
Model                                                       
Blend (4 models)                   120.9696  45.0312  0.7030
CatBoost d8 + XGBoost 50/50        121.1308  44.6756  0.7022
Old blend (2 CatBoost + LightGBM)  121.7812  46.4170  0.6990
CatBoost d8 alone                  122.8713  47.5475  0.6936
XGBoost alone                      123.5410  45.3110  0.6902

Per-fold R2, blend: [0.651, 0.798, 0.785, 0.78, 0.583]
Residual correlation between models (lower = more useful to blend):
             CatBoost d8  CatBoost d6  XGBoost  LightGBM
CatBoost d8        1.000        0.989    0.933     0.928
CatBoost d6        0.989        1.000    0.923     0.922
XGBoost            0.933        0.923    1.000     0.941
LightGBM           0.928        0.922    0.941     1.000


In [14]:
test_pred = {}
for name in MODELS:
    t = time.time()
    test_pred[name] = fit_predict(name, X, y, Xt)
    print(f"{name:12s} fitted on full data ({time.time()-t:.0f}s)")

final = np.clip(blend(test_pred), 0, None)

submission = sample_sub.copy()
submission["price"] = final
assert submission.shape == (len(test), 2) and list(submission.columns) == ["id", "price"]
assert submission["id"].tolist() == test["id"].tolist(), "ID order changed!"
assert submission["price"].notna().all()

SAVE_PATH = f"{OUT_DIR}/submission_xgb_catboost_blend.csv"
submission.to_csv(SAVE_PATH, index=False)
print("saved:", SAVE_PATH)
print(submission["price"].describe().round(2))

CatBoost d8  fitted on full data (42s)
CatBoost d6  fitted on full data (29s)
XGBoost      fitted on full data (20s)
LightGBM     fitted on full data (4s)
saved: /content/drive/MyDrive/ML_project_dataset/submission_xgb_catboost_blend.csv
count    4520.00
mean      280.37
std       188.25
min        13.71
25%       158.41
50%       246.90
75%       347.44
max      2235.04
Name: price, dtype: float64


In [15]:

# Residual analysis for the four-model OOF blend
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

diag = train.copy().reset_index(drop=True)
diag["oof_pred"] = np.asarray(blend_oof)
diag["error"] = diag["price"] - diag["oof_pred"]
diag["abs_error"] = diag["error"].abs()

def error_summary(df, group_col):
    result = df.groupby(group_col, observed=True).apply(
        lambda g: pd.Series({
            "rows": len(g),
            "MAE": mean_absolute_error(g["price"], g["oof_pred"]),
            "RMSE": np.sqrt(mean_squared_error(g["price"], g["oof_pred"])),
            "mean_error": g["error"].mean()
        }),
        include_groups=False
    )
    return result.round(2).sort_values("MAE", ascending=False)

# 1. Errors by flight class
print("=== ERROR BY FLIGHT CLASS ===")
display(error_summary(diag, "flight_class"))

# 2. Errors by actual ticket price
diag["price_band"] = pd.cut(
    diag["price"],
    bins=[-1, 100, 200, 350, 600, 1000, float("inf")],
    labels=["0–100", "100–200", "200–350",
            "350–600", "600–1000", "1000+"]
)
print("=== ERROR BY ACTUAL PRICE BAND ===")
display(error_summary(diag, "price_band"))

# 3. Errors by days left until departure
diag["days_left_band"] = pd.cut(
    diag["days_left"],
    bins=[-1, 7, 14, 21, 30, 45, float("inf")],
    labels=["0–7", "8–14", "15–21", "22–30", "31–45", "46+"]
)
print("=== ERROR BY DAYS LEFT ===")
display(error_summary(diag, "days_left_band"))

# 4. Twenty largest prediction errors
feature_cols = [
    "id", "airline", "flight", "source_city", "destination_city",
    "flight_class", "stops", "duration", "days_left", "price",
    "oof_pred", "error", "abs_error"
]
print("=== 20 LARGEST OOF ERRORS ===")
display(
    diag.nlargest(20, "abs_error")[feature_cols]
    .round({"oof_pred": 2, "error": 2, "abs_error": 2})
)

# Save OOF predictions so you don't have to retrain just for diagnostics
save_cols = ["id", "price", "oof_pred", "error", "abs_error"]
for name, pred in oof.items():
    diag[name + "_oof"] = np.asarray(pred)
    save_cols.append(name + "_oof")

oof_path = f"{DATA_DIR}/oof_diagnostics.csv"
diag[save_cols].to_csv(oof_path, index=False)
print("Saved OOF diagnostics to:", oof_path)


=== ERROR BY FLIGHT CLASS ===


,rows,MAE,RMSE,mean_error
flight_class,,,,
Business,7365.0,77.47,182.92,2.46
Economy,10711.0,22.73,41.10,0.31


=== ERROR BY ACTUAL PRICE BAND ===


,rows,MAE,RMSE,mean_error
price_band,,,,
1000+,191.0,527.30,949.96,390.08
600–1000,937.0,134.00,201.44,82.08
350–600,2843.0,63.55,93.88,26.51
200–350,7538.0,31.31,47.98,-13.34
100–200,4389.0,27.11,41.34,-13.19
0–100,2178.0,23.91,44.03,-21.52


=== ERROR BY DAYS LEFT ===


,rows,MAE,RMSE,mean_error
days_left_band,,,,
22–30,3657.0,52.43,109.60,2.97
31–45,4567.0,48.41,131.08,-0.35
0–7,3273.0,43.78,168.03,4.52
15–21,3246.0,40.59,82.40,-2.07
8–14,3333.0,37.83,91.11,1.23


=== 20 LARGEST OOF ERRORS ===


,id,airline,flight,source_city,destination_city,flight_class,stops,duration,days_left,price,oof_pred,error,abs_error
15787,16,United,UA 729,Los Angeles,San Diego,Business,one,9.37,1,8996,914.80,8081.20,8081.20
15200,22275,Alaska_Airlines,AS 2224,San Diego,San Jose,Business,one,14.52,43,5564,572.26,4991.74,4991.74
5936,19925,Alaska_Airlines,AS 668,Los Angeles,San Jose,Business,one,5.58,37,3607,577.32,3029.68,3029.68
7528,4233,United,UA 1100,San Diego,San Jose,Business,one,13.27,9,3029,1164.88,1864.12,1864.12
5972,14738,American_Airlines,AA 6432,Los Angeles,San Francisco,Business,zero,1.62,28,2025,257.96,1767.04,1767.04
1573,14736,American_Airlines,AA 6277,Los Angeles,San Francisco,Business,zero,1.77,28,2025,323.82,1701.18,1701.18
3740,14737,American_Airlines,AA 2072,Los Angeles,San Francisco,Business,zero,1.58,28,2025,386.49,1638.51,1638.51
10611,5522,United,UA 2013,San Jose,Sacramento,Business,one,6.28,10,3194,1557.50,1636.50,1636.50
11239,15011,American_Airlines,AA 6276,San Francisco,Los Angeles,Business,zero,1.62,26,2025,390.99,1634.01,1634.01
16942,19956,American_Airlines,AA 6276,San Francisco,Los Angeles,Business,zero,1.62,38,2025,401.96,1623.04,1623.04


Saved OOF diagnostics to: /content/drive/MyDrive/ML_project_dataset/oof_diagnostics.csv
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [16]:

# Zero-price rows
zero_rows = train[train["price"] == 0].sort_values(
    ["airline", "source_city", "destination_city", "days_left"]
)

print("Zero-price rows:", len(zero_rows))
display(zero_rows)

# Highest-priced rows
print("=== 30 HIGHEST PRICES ===")
display(train.nlargest(30, "price").sort_values("price", ascending=False))

# Price counts and upper percentiles
print("=== PRICE DISTRIBUTION ===")
display(
    train["price"].describe(
        percentiles=[.25, .50, .75, .90, .95, .99, .995, .999]
    ).to_frame("price")
)

# Check whether exact duplicate feature combinations have different prices
features = [
    c for c in train.columns
    if c not in ["id", "price"]
]
dup_features = train[
    train.duplicated(subset=features, keep=False)
].sort_values(features)

print("Rows sharing identical flight features:", len(dup_features))
display(dup_features.head(50))


Zero-price rows: 25


,id,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,flight_class,duration,days_left,price
5991,17620,Alaska_Airlines,AS 490,Sacramento,Early_Morning,one,Morning,San Jose,Economy,4.33,35,0
13347,22151,Alaska_Airlines,AS 755,San Diego,Early_Morning,one,Morning,Los Angeles,Business,4.05,44,0
8252,636,Southwest,WN 3684,Sacramento,Morning,one,Afternoon,San Jose,Economy,3.08,1,0
4350,3307,Southwest,WN 2864,Sacramento,Morning,one,Morning,San Jose,Economy,3.08,6,0
3101,5689,Southwest,WN 3687,Sacramento,Evening,one,Night,San Jose,Economy,3.08,9,0
6642,4923,Southwest,WN 2864,Sacramento,Morning,one,Morning,San Jose,Economy,3.08,10,0
7640,7770,Southwest,WN 2864,Sacramento,Morning,one,Morning,San Jose,Economy,3.08,13,0
5386,7879,Southwest,WN 3684,Sacramento,Morning,one,Afternoon,San Jose,Economy,3.08,15,0
7852,11219,Southwest,WN 3949,Sacramento,Afternoon,one,Evening,San Jose,Economy,3.08,20,0
14049,12816,Southwest,WN 3903,Sacramento,Afternoon,one,Evening,San Jose,Economy,3.17,24,0


=== 30 HIGHEST PRICES ===


,id,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,flight_class,duration,days_left,price
15787,16,United,UA 729,Los Angeles,Late_Night,one,Morning,San Diego,Business,9.37,1,8996
15200,22275,Alaska_Airlines,AS 2224,San Diego,Night,one,Afternoon,San Jose,Business,14.52,43,5564
5936,19925,Alaska_Airlines,AS 668,Los Angeles,Early_Morning,one,Afternoon,San Jose,Business,5.58,37,3607
2473,13008,United,UA 1055,Sacramento,Morning,one,Evening,San Jose,Business,6.70,23,3204
12601,11469,United,UA 1055,Sacramento,Morning,one,Evening,San Jose,Business,6.70,19,3204
4249,7975,United,UA 1055,Sacramento,Morning,one,Evening,San Jose,Business,6.70,14,3194
10611,5522,United,UA 2013,San Jose,Morning,one,Afternoon,Sacramento,Business,6.28,10,3194
12674,5650,United,UA 1023,Sacramento,Morning,one,Evening,San Jose,Business,6.70,10,3194
7528,4233,United,UA 1100,San Diego,Night,one,Morning,San Jose,Business,13.27,9,3029
12109,19085,United,UA 2013,San Jose,Morning,one,Afternoon,Sacramento,Business,6.43,37,2424


=== PRICE DISTRIBUTION ===


,price
count,18076.000000
mean,280.264550
std,221.965133
min,0.000000
25%,157.000000
50%,234.000000
75%,336.000000
90%,519.000000
95%,656.250000
99%,1016.000000


Rows sharing identical flight features: 12


,id,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,flight_class,duration,days_left,price
1595,4625,Alaska_Airlines,AS 776,San Francisco,Afternoon,two_or_more,Night,Sacramento,Economy,9.63,6,368
12222,4626,Alaska_Airlines,AS 776,San Francisco,Afternoon,two_or_more,Night,Sacramento,Economy,9.63,6,368
10605,5579,Alaska_Airlines,AS 776,San Francisco,Afternoon,two_or_more,Night,Sacramento,Economy,9.63,10,288
11894,5580,Alaska_Airlines,AS 776,San Francisco,Afternoon,two_or_more,Night,Sacramento,Economy,9.63,10,288
2849,11942,Southwest,WN 4343,San Jose,Morning,two_or_more,Late_Night,Sacramento,Economy,13.00,22,343
11374,11940,Southwest,WN 4343,San Jose,Morning,two_or_more,Late_Night,Sacramento,Economy,13.00,22,343
6337,16552,United,UA 444,San Francisco,Morning,two_or_more,Night,Sacramento,Economy,11.90,29,173
8864,16551,United,UA 444,San Francisco,Morning,two_or_more,Night,Sacramento,Economy,11.90,29,173
5871,6639,United,UA 6021,Sacramento,Morning,two_or_more,Night,San Francisco,Economy,11.22,13,324
16192,6638,United,UA 6021,Sacramento,Morning,two_or_more,Night,San Francisco,Economy,11.22,13,324


In [19]:

# Inspect the most extreme prices and zero-price records
cols = [
    "id", "airline", "flight", "source_city",
    "destination_city", "flight_class", "stops",
    "duration", "days_left", "price"
]

print("Highest 15 prices:")
display(train.nlargest(15, "price")[cols])

print("Zero-price rows by airline and route:")
display(
    train[train["price"] == 0]
    .groupby(
        ["airline", "source_city", "destination_city", "flight_class"]
    )
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)


Highest 15 prices:


,id,airline,flight,source_city,destination_city,flight_class,stops,duration,days_left,price
15787,16,United,UA 729,Los Angeles,San Diego,Business,one,9.37,1,8996
15200,22275,Alaska_Airlines,AS 2224,San Diego,San Jose,Business,one,14.52,43,5564
5936,19925,Alaska_Airlines,AS 668,Los Angeles,San Jose,Business,one,5.58,37,3607
2473,13008,United,UA 1055,Sacramento,San Jose,Business,one,6.70,23,3204
12601,11469,United,UA 1055,Sacramento,San Jose,Business,one,6.70,19,3204
4249,7975,United,UA 1055,Sacramento,San Jose,Business,one,6.70,14,3194
10611,5522,United,UA 2013,San Jose,Sacramento,Business,one,6.28,10,3194
12674,5650,United,UA 1023,Sacramento,San Jose,Business,one,6.70,10,3194
7528,4233,United,UA 1100,San Diego,San Jose,Business,one,13.27,9,3029
12109,19085,United,UA 2013,San Jose,Sacramento,Business,one,6.43,37,2424


Zero-price rows by airline and route:


,airline,source_city,destination_city,flight_class,count
3,Southwest,San Jose,Sacramento,Economy,13
2,Southwest,Sacramento,San Jose,Economy,10
1,Alaska_Airlines,San Diego,Los Angeles,Business,1
0,Alaska_Airlines,Sacramento,San Jose,Economy,1


In [20]:
o[va_i] = fit_predict(name, X.iloc[tr_i], y.iloc[tr_i], X.iloc[va_i])

In [21]:
# Remove zero-price records from this fold's training data only.
# Keep the validation fold unchanged for a fair comparison.
train_idx = tr_i[y.iloc[tr_i].to_numpy() > 0]

o[va_i] = fit_predict(
    name,
    X.iloc[train_idx],
    y.iloc[train_idx],
    X.iloc[va_i]
)

In [1]:
import pandas as pd
from IPython.display import display

holdout_results = [
    {"Model": "CatBoost depth 8", "Evaluation": "80/20 holdout", "RMSE": 140.7829, "MAE": 49.3047, "R2": 0.6477},
    {"Model": "CatBoost depth 6", "Evaluation": "80/20 holdout", "RMSE": 141.5311, "MAE": 50.7065, "R2": 0.6439},
    {"Model": "XGBoost", "Evaluation": "80/20 holdout", "RMSE": 141.2646, "MAE": 47.0346, "R2": 0.6453},
    {"Model": "LightGBM", "Evaluation": "80/20 holdout", "RMSE": 144.1350, "MAE": 49.7454, "R2": 0.6307},
    {"Model": "Random Forest (earlier notebook)", "Evaluation": "earlier holdout; reference", "RMSE": 144.7390, "MAE": 47.0170, "R2": 0.6276},
    {"Model": "Polynomial degree 8 (baseline)", "Evaluation": "earlier holdout; reference", "RMSE": 176.2600, "MAE": 86.4350, "R2": 0.4477},
]
print("HOLDOUT VALIDATION")
display(pd.DataFrame(holdout_results).sort_values("RMSE").round(4))

oof_results = [
    {"Model": "CatBoost depth 8", "Evaluation": "5-fold OOF", "RMSE": 122.8713, "MAE": 47.5475, "R2": 0.6936},
    {"Model": "CatBoost depth 6", "Evaluation": "5-fold OOF", "RMSE": 123.3718, "MAE": 48.8076, "R2": 0.6911},
    {"Model": "XGBoost", "Evaluation": "5-fold OOF", "RMSE": 123.5410, "MAE": 45.3110, "R2": 0.6902},
    {"Model": "LightGBM", "Evaluation": "5-fold OOF", "RMSE": 125.8401, "MAE": 47.4719, "R2": 0.6786},
    {"Model": "Four-model blend (0.35/0.20/0.30/0.15)", "Evaluation": "5-fold OOF", "RMSE": 120.9696, "MAE": 45.0312, "R2": 0.7030},
    {"Model": "CatBoost depth 8 + XGBoost (50/50)", "Evaluation": "5-fold OOF", "RMSE": 121.1308, "MAE": 44.6756, "R2": 0.7022},
    {"Model": "Two CatBoost + LightGBM blend (0.40/0.30/0.30)", "Evaluation": "5-fold OOF", "RMSE": 121.7812, "MAE": 46.4170, "R2": 0.6990},
]
print("FIVE-FOLD OOF")
display(pd.DataFrame(oof_results).sort_values("RMSE").round(4))

HOLDOUT VALIDATION


,Model,Evaluation,RMSE,MAE,R2
0,CatBoost depth 8,80/20 holdout,140.7829,49.3047,0.6477
2,XGBoost,80/20 holdout,141.2646,47.0346,0.6453
1,CatBoost depth 6,80/20 holdout,141.5311,50.7065,0.6439
3,LightGBM,80/20 holdout,144.1350,49.7454,0.6307
4,Random Forest (earlier notebook),earlier holdout; reference,144.7390,47.0170,0.6276
5,Polynomial degree 8 (baseline),earlier holdout; reference,176.2600,86.4350,0.4477


FIVE-FOLD OOF


,Model,Evaluation,RMSE,MAE,R2
4,Four-model blend (0.35/0.20/0.30/0.15),5-fold OOF,120.9696,45.0312,0.7030
5,CatBoost depth 8 + XGBoost (50/50),5-fold OOF,121.1308,44.6756,0.7022
6,Two CatBoost + LightGBM blend (0.40/0.30/0.30),5-fold OOF,121.7812,46.4170,0.6990
0,CatBoost depth 8,5-fold OOF,122.8713,47.5475,0.6936
1,CatBoost depth 6,5-fold OOF,123.3718,48.8076,0.6911
2,XGBoost,5-fold OOF,123.5410,45.3110,0.6902
3,LightGBM,5-fold OOF,125.8401,47.4719,0.6786
